In [1]:
%pip install mlflow
%pip install --upgrade langchain
%pip install faiss-cpu
%pip install pydantic==1.10.9
%pip install chromadb
%pip install InstructorEmbedding
%pip install gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.4/28.4 MB 3.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 4.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.6/134.6 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.3/106.3 kB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 4.0 MB/s eta 0:00:0000:0100:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 4.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 3.8 MB/s eta 0:00:00

In [3]:
pip install -U langchain-community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 4.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.3 MB/s eta 0:00:00
  Using cached typing_inspect-0.9.0-py3-none-any.whl (8.8 kB)
  Using cached mypy_extensions-1.0.0-py3-none-any.whl (4.7 kB)

[notice] A new release of pip available: 22.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
from langchain.document_loaders.csv_loader import CSVLoader
loader = CSVLoader(file_path="/Workspace/DataforRAG_final1.csv",
encoding="utf-8", csv_args={'delimiter': ','})
data = loader.load()


from langchain.text_splitter import RecursiveCharacterTextSplitter
#from langchain.text_splitter import CharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
text_chunks = text_splitter.split_documents(data)


RuntimeError: Error loading /Workspace/DataforRAG_final1.csv

In [ ]:
from langchain.embeddings import HuggingFaceInstructEmbeddings
instructor_embeddings = HuggingFaceInstructEmbeddings(model_name="hkunlp/instructor-xl", 
                                                    model_kwargs={"device": "cuda"})

In [ ]:
# Embed and store the texts
# Supplying a persist_directory will store the embeddings on disk
from langchain.vectorstores import Chroma
persist_directory = 'db'

## Here is the nmew embeddings being used
embedding = instructor_embeddings

vectordb = Chroma.from_documents(documents=text_chunks, 
                                 embedding=embedding,
                                 persist_directory=persist_directory)


# persiste the db to disk
vectordb.persist()
vectordb = None

# Now we can load the persisted database from disk, and use it as normal. 
vectordb = Chroma(persist_directory=persist_directory, 
                  embedding_function=embedding)


In [ ]:
from langchain.llms import MlflowAIGateway
from langchain.prompts import PromptTemplate
from langchain.chains import RetrievalQA
import gradio as gr
from mlflow.gateway import set_gateway_uri, create_route, query,delete_route

set_gateway_uri("databricks")

mosaic_completion_route = MlflowAIGateway(
  gateway_uri="databricks",
  route="completion"
)

# Wrap the prompt and Gateway Route into a chain

template = """[INST] <>
You are Banking Question Answering Machine. Answer Accordingly
<>

{context}

{question} [/INST]

"""
prompt = PromptTemplate(input_variables=['context', 'question'], 
template=template)

retrieval_qa_chain = RetrievalQA.from_chain_type(llm=mosaic_completion_route, chain_type="stuff", 
retriever=vectordb.as_retriever(), chain_type_kwargs={"prompt": prompt})


def generating_text(prompt,route_param,token):
    # Create a Route for text completions with MosaicML Inference API 
    create_route(
        name=route_param,
        route_type="llm/v1/completions",
        model={
            "name": "llama2-70b-chat",
            "provider": "mosaicml",
            "mosaicml_config": {
            "mosaicml_api_key": "3abc"
            }
        }
    )

    response1 = query(
    route=route_param,
    data={"prompt": prompt,"temperature": 0.1,
        "max_tokens": token}
    )

    return(response1)

In [ ]:
import string
import random
 
# initializing size of string
N = 7

def greet(Input,chat_history):
    RouteName1="text1"
    RouteName2="text2"
    system="""you are a translator which converts english to hindi. 
    Please translate the given text to hindi language and 
    only return the content translated. no explanation"""
    
    system1="""you are a translator which converts hindi to english. 
    Please translate the given text to english language from hindi language and 
    only return the content translated. 
    no explanation"""
    
    prompt=f"[INST] <> {system1} <> {Input}[/INST]"
    delete_route("text1")
    result=generating_text(prompt,RouteName1,400)
    res=result['candidates'][0]['text']
    t=retrieval_qa_chain.run(res)
    prompt2=f"[INST] <> {system} <> {t} [/INST]"
    delete_route("text2")
    token=800
    result1=generating_text(prompt2,RouteName2,token)
    chat_history.append((Input, result1['candidates'][0]['text']))
    return "", chat_history


with gr.Blocks(theme=gr.themes.Soft(primary_hue=gr.themes.colors.blue, 
    secondary_hue=gr.themes.colors.red)) as demo:
    gr.Markdown("## सखा- भाषा अब कोई बाधा नहीं है")
    chatbot = gr.Chatbot(height=400) #just to fit the notebook
    msg = gr.Textbox(label="Prompt",placeholder="अपना प्रश्न हिंदी में यहां दर्ज करें",max_lines=2)
    with gr.Row():
        btn = gr.Button("Submit")
        clear = gr.ClearButton(components=[msg, chatbot], value="Clear console")
    # btn = gr.Button("Submit")
    # clear = gr.ClearButton(components=[msg, chatbot], value="Clear console")
    btn.click(greet, inputs=[msg, chatbot], outputs=[msg, chatbot])
    msg.submit(greet, inputs=[msg, chatbot], outputs=[msg, chatbot])
    gr.Examples([["एचडीएफसी बैंक का कस्टमर केयर नंबर क्या है?"],
                 ["गोल्ड लोन क्या है??"],['गोल्ड लोन के लिए आवश्यक दस्तावेज।']], 
                 inputs=[msg,chatbot])

gr.close_all()
demo.launch(share=True,debug=True)

     #import csv